In [1]:
from dotenv import load_dotenv

load_dotenv()


True

## Setup Tools


In [5]:
import asyncio

from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent

RETRYABLE_MCP_CODES = {-32603}

class RetryMCPInterceptor:
    """Intercept MCP tool calls: retry transient failures, surface all errors gracefully.

    - Retryable McpError codes (e.g. -32603): retry with exponential backoff.
    - Non-retryable McpError codes (e.g. -32602): return error message immediately.
    - Any other exception (fetch failed, network errors, etc.): retry then return error message.
    """

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(code {exc.error.code}, attempt {attempt+1}/{self.max_retries}): {exc}")
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(
                        content=[TextContent(type="text", text=f"Tool call failed (non-retryable): {exc}")],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(attempt {attempt+1}/{self.max_retries}): {exc}")

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)

        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

tools = await client.get_tools()

In [6]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters     
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

In [7]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

## Setup RAG (Wedding Planning Knowledge Base)

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# Create sample wedding planning documents (in a real scenario, load from PDFs)
wedding_docs = [
    Document(page_content="""
    Wedding Planning Timeline:
    12 months before: Book venue, photographer, and caterer
    9 months before: Send save-the-dates, book florist and DJ
    6 months before: Order wedding dress, book hair and makeup
    3 months before: Send invitations, finalize menu
    1 month before: Final dress fitting, create seating chart
    1 week before: Confirm all vendors, rehearsal dinner
    """, metadata={"source": "timeline"}),
    
    Document(page_content="""
    Destination Wedding Tips:
    - Consider travel time and costs for guests
    - Book venue 12-18 months in advance
    - Spring (April-June) and Fall (September-November) are ideal for European destinations
    - Paris weddings: Best in May, September, or October for weather
    - Egypt weddings: Best in November-March to avoid extreme heat
    - Provide welcome bags with local maps and recommendations
    """, metadata={"source": "destination"}),
    
    Document(page_content="""
    Wedding Budget Guidelines:
    Venue & Catering: 40-50% of total budget
    Photography & Videography: 10-15%
    Music & Entertainment: 8-10%
    Flowers & Decorations: 8-10%
    Wedding Attire: 5-8%
    Invitations & Stationery: 2-3%
    Wedding Cake: 2-3%
    Transportation: 2-3%
    Contingency: 5-10%
    """, metadata={"source": "budget"}),
    
    Document(page_content="""
    Guest Count Planning:
    Small wedding: 20-50 guests (intimate, easier to manage)
    Medium wedding: 50-150 guests (most common size)
    Large wedding: 150+ guests (requires larger venue, more coordination)
    
    For 100 guests:
    - Book venue with capacity for 120-130 (account for staff, vendors)
    - Plan 1 restroom per 50 guests
    - Arrange seating in tables of 8-10
    - Budget approximately $100-200 per guest for food and drinks
    """, metadata={"source": "guests"}),
    
    Document(page_content="""
    Music & Entertainment Guide:
    Jazz weddings: Perfect for cocktail hour and dinner, sophisticated atmosphere
    - Hire live jazz trio or quartet
    - Great for Paris, New Orleans, or urban venues
    - Budget: $1,500-4,000 for 3-4 hours
    
    Classical: Elegant, traditional, ceremony music
    Pop/Top 40: High energy, dance-focused reception
    Live band vs DJ: Bands cost more ($3,000-10,000) but create ambiance
    """, metadata={"source": "music"})
]

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=100
)
all_splits = text_splitter.split_documents(wedding_docs)

# Create embeddings and vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore(embeddings)
ids = vector_store.add_documents(documents=all_splits)

print(f"Added {len(all_splits)} document chunks to vector store")

In [ ]:
@tool
def search_wedding_knowledge(query: str) -> str:
    """Search the wedding planning knowledge base for expert advice on timelines, budgets, destinations, and best practices."""
    results = vector_store.similarity_search(query, k=2)
    # Combine top 2 results
    return "\n\n".join([doc.page_content for doc in results])

## Create State

In [9]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## Create Subagents


In [31]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os

# Travel agent
travel_agent = create_agent(
    model=init_chat_model(
        model="llama-3.1-8b-instant",
        model_provider="groq",
        api_key=os.getenv("GROQ_API_KEY"),
        temperature=0,
    ),
    tools=tools,
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location.
    
    CRITICAL DATE REQUIREMENT:
    - Today's date is May 21, 2026 (21/05/2026)
    - You MUST use departure dates that are in the future
    - Use dates in format: 2026-08-15 or 2026-09-01 or 2026-10-10
    - Minimum: 3 months from today (August 2026 or later)
    - Example valid dates: 2026-08-15, 2026-09-20, 2026-10-05
    
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location, but must be in the future)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    If the MCP tool fails, returns malformed output, or does not give you usable flight results, try the tool again with a different date.
    If you get an error about dates being in the past, use a date at least 6 months in the future (November 2026).
    Once you have found the best options, let the user know your shortlist of options.
    """
)

In [32]:
# Venue agent
venue_agent = create_agent(
    model=init_chat_model(
        model="llama-3.1-8b-instant",
        model_provider="groq",
        api_key=os.getenv("GROQ_API_KEY"),
        temperature=0,
    ),
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options. 
    You have a suggested limit of 5 web searches. Count every web_search call you make.
    After 5 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

In [12]:
# Playlist agent
playlist_agent = create_agent(
    model=init_chat_model(
        model="llama-3.1-8b-instant",
        model_provider="groq",
        api_key=os.getenv("GROQ_API_KEY"),
        temperature=0,
    ),
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

In [ ]:
# Wedding Planning Expert (RAG agent)
wedding_expert = create_agent(
    model=init_chat_model(
        model="llama-3.1-8b-instant",
        model_provider="groq",
        api_key=os.getenv("GROQ_API_KEY"),
        temperature=0,
    ),
    tools=[search_wedding_knowledge],
    system_prompt="""
    You are a wedding planning expert with access to comprehensive wedding planning knowledge.
    Search the knowledge base to provide expert advice on:
    - Wedding planning timelines and schedules
    - Budget allocation and cost estimates
    - Destination wedding best practices
    - Guest count planning and logistics
    - Music and entertainment recommendations
    
    Always search the knowledge base first before providing advice.
    Provide specific, actionable recommendations based on the retrieved information.
    """
)

## Main Coordinator


In [33]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"Find flights from {origin} to {destination}")]})
    return response['messages'][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def get_expert_advice(runtime: ToolRuntime) -> str:
    """Wedding planning expert provides advice on timelines, budgets, and best practices."""
    destination = runtime.state["destination"]
    guest_count = runtime.state["guest_count"]
    query = f"What are the best practices and recommendations for planning a wedding in {destination} for {guest_count} guests?"
    response = wedding_expert.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


In [14]:
coordinator = create_agent(
    model=init_chat_model(
        model="llama-3.1-8b-instant",
        model_provider="groq",
        api_key=os.getenv("GROQ_API_KEY"),
        temperature=0,
    ),
    tools=[search_flights, search_venues, suggest_playlist, get_expert_advice, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator. 
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks 
    to your specialists for flights, venues, playlists, and expert wedding planning advice.
    The expert can provide advice on timelines, budgets, destination best practices, and guest planning.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)


## Test RAG Agent

In [ ]:
# Test the RAG agent standalone
test_response = wedding_expert.invoke({
    "messages": [HumanMessage(content="What's the best time of year for a wedding in Paris?")]
})

print(test_response['messages'][-1].content)

## Test


In [ ]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from Egypt and I'd like a wedding in Paris for 100 guests, jazz-genre")],
    },
    config={"tags": ["WP"], "recursion_limit": 40},  #tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
)

In [ ]:
from pprint import pprint

pprint(response)

In [ ]:
print(response["messages"][-1].content)